# 06 — Sentiment Scoring with FinBERT
Score Reddit posts and news headlines, aggregate to a daily sentiment index.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax
import os

plt.rcParams['figure.dpi'] = 120
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print('Device:', DEVICE)

## 1. Load FinBERT
`ProsusAI/finbert` outputs 3 classes: positive, negative, neutral.


In [ ]:
MODEL_NAME = 'ProsusAI/finbert'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
finbert    = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)
finbert.eval()
print('FinBERT loaded. Labels:', finbert.config.id2label)

## 2. Scoring function

In [ ]:
def score_texts(texts: list, batch_size: int = 32) -> pd.DataFrame:
    """
    Returns DataFrame with columns: positive, negative, neutral, sentiment_score.
    sentiment_score = positive - negative  (range: -1 to +1)
    """
    all_pos, all_neg, all_neu = [], [], []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=512, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            logits = finbert(**enc).logits
        probs = softmax(logits, dim=-1).cpu().numpy()
        # FinBERT label order: positive=0, negative=1, neutral=2
        all_pos.extend(probs[:, 0])
        all_neg.extend(probs[:, 1])
        all_neu.extend(probs[:, 2])

    return pd.DataFrame({
        'positive':        all_pos,
        'negative':        all_neg,
        'neutral':         all_neu,
        'sentiment_score': np.array(all_pos) - np.array(all_neg),
    })


# Quick sanity check
test_texts = [
    'Silver is going to the moon! Buy buy buy!',
    'Silver prices crash amid rising dollar and Fed hawkishness.',
    'Silver traded flat today with low volume.',
]
score_texts(test_texts)

## 3. Score Reddit posts

In [ ]:
reddit_path = '../data/raw/reddit_history.csv'

if os.path.exists(reddit_path):
    reddit = pd.read_csv(reddit_path, parse_dates=['created_utc'])
    reddit = reddit.dropna(subset=['title'])

    # Combine title + first 100 chars of body for more signal
    reddit['text'] = reddit['title'] + '. ' + reddit['selftext'].fillna('').str[:100]

    print(f'Scoring {len(reddit):,} Reddit posts...')
    scores = score_texts(reddit['text'].tolist())

    reddit = pd.concat([reddit.reset_index(drop=True), scores], axis=1)
    reddit['date'] = pd.to_datetime(reddit['created_utc']).dt.date

    # Weight by score (upvotes) — high-engagement posts matter more
    reddit['weight'] = reddit['score'].clip(lower=1)
    reddit['weighted_sentiment'] = reddit['sentiment_score'] * reddit['weight']

    daily_reddit = (
        reddit.groupby('date')
        .agg(
            reddit_sentiment=('weighted_sentiment', 'sum'),
            reddit_weight_sum=('weight', 'sum'),
            reddit_post_count=('sentiment_score', 'count'),
        )
        .assign(reddit_sentiment=lambda d: d['reddit_sentiment'] / d['reddit_weight_sum'])
    )
    daily_reddit.index = pd.to_datetime(daily_reddit.index)
    print(daily_reddit.tail())
else:
    print('No Reddit history file — run collect_reddit.py first.')
    daily_reddit = pd.DataFrame()

## 4. Score news headlines

In [ ]:
news_path = '../data/raw/news_gdelt.csv'

if os.path.exists(news_path):
    news = pd.read_csv(news_path, parse_dates=['datetime'])
    news = news.dropna(subset=['title'])

    print(f'Scoring {len(news):,} news headlines...')
    news_scores = score_texts(news['title'].tolist())

    news = pd.concat([news.reset_index(drop=True), news_scores], axis=1)
    news['date'] = pd.to_datetime(news['datetime']).dt.date

    daily_news = (
        news.groupby('date')['sentiment_score']
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'news_sentiment', 'count': 'news_article_count'})
    )
    daily_news.index = pd.to_datetime(daily_news.index)
    print(daily_news.tail())
else:
    print('No GDELT news file — run collect_news.py first.')
    daily_news = pd.DataFrame()

## 5. Combine into daily sentiment index

In [ ]:
sentiment_frames = [f for f in [daily_reddit, daily_news] if not f.empty]

if sentiment_frames:
    daily_sentiment = pd.concat(sentiment_frames, axis=1).sort_index()

    # Composite score: equal weight of Reddit + news (can tune later)
    cols = [c for c in ['reddit_sentiment', 'news_sentiment'] if c in daily_sentiment.columns]
    daily_sentiment['sentiment_score'] = daily_sentiment[cols].mean(axis=1)

    # Forward-fill weekends/gaps (no posts on weekends)
    daily_sentiment = daily_sentiment.resample('B').ffill()  # business days

    os.makedirs('../data/processed', exist_ok=True)
    daily_sentiment.to_csv('../data/processed/daily_sentiment.csv')
    print('Saved:', daily_sentiment.shape)
    daily_sentiment.head()
else:
    print('No sentiment data to combine yet.')

## 6. Visualise sentiment vs silver price

In [ ]:
if sentiment_frames:
    prices = pd.read_csv('../data/raw/daily_prices.csv', index_col=0, parse_dates=True)
    prices.index = prices.index.tz_localize(None)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

    ax1.plot(prices['silver'], lw=1, color='steelblue')
    ax1.set_ylabel('Silver price (USD/oz)')
    ax1.set_title('Silver Price vs Daily Sentiment Index')

    ax2.fill_between(daily_sentiment.index,
                     daily_sentiment['sentiment_score'],
                     alpha=0.5, color='darkorange')
    ax2.axhline(0, color='black', lw=0.8)
    ax2.set_ylabel('Sentiment score')

    plt.tight_layout()
    plt.show()